# 2. Real data with scikit-learn: pipelines, cross-validation, grid search

The Titanic passenger list is small, familiar and messy: it mixes numbers
with categories, has missing ages, and its rules are easy to judge. This
notebook downloads it through scikit-learn (cached in your home directory
after the first run) and shows how the fuzzy classifier fits into the usual
scikit-learn workflow.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

from ex_fuzzy import BaseFuzzyRulesClassifier, utils

titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
X = titanic.frame[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
y = np.where(titanic.frame['survived'].astype(int) == 1, 'survived', 'died')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X.dtypes.to_string())
print('\nmissing values per column:')
print(X.isna().sum().to_string())

pclass         int64
sex         category
age          float64
sibsp          int64
parch          int64
fare         float64
embarked    category

missing values per column:
pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2


## Missing values are refused, not guessed

A missing value has no membership degree, so the classifier rejects it and
tells you which columns are affected. Imputation is a modelling decision,
and it belongs in a pipeline step you control.

In [2]:
try:
    BaseFuzzyRulesClassifier(n_gen=1).fit(X_train, y_train)
except ValueError as error:
    print(error)

X contains missing or infinite values in columns [2, 5]. Fuzzy rule classifiers need complete feature values: impute them first, or use FERL with an observed mask for missing features.


## A pipeline with imputation

`SimpleImputer` with pandas output keeps the column names, which become the
variable names in the rules. Categorical columns (`sex`, `embarked`, and
`pclass`, which only takes three whole numbers) are detected automatically
and get one crisp set per category; the numerical ones get fuzzy partitions
that the genetic search optimises.

In [3]:
pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent').set_output(transform='pandas')),
    ('fuzzy', BaseFuzzyRulesClassifier(nRules=8, nAnts=3, n_gen=30, pop_size=30, random_state=0)),
])
pipeline.fit(X_train, y_train)
print(f'train accuracy: {pipeline.score(X_train, y_train):.3f}')
print(f'test accuracy:  {pipeline.score(X_test, y_test):.3f}')

train accuracy: 0.779
test accuracy:  0.817


In [4]:
for variable in pipeline['fuzzy'].lvs:
    print(f'{variable.name:10s} {variable.linguistic_variable_names()}')

pclass     ['1', '2', '3']
sex        ['female', 'male']
age        ['Low', 'Medium', 'High']
sibsp      ['Low', 'Medium', 'High']
parch      ['Low', 'Medium', 'High']
fare       ['Low', 'Medium', 'High']
embarked   ['C', 'Q', 'S']


In [5]:
pipeline['fuzzy'].print_rules()

Rules for consequent: died
----------------
IF sex IS female AND sibsp IS Medium AND parch IS Low WITH DS 0.0046960275726492225, ACC 0.6842105263157895
IF sex IS male WITH DS 0.41331614783449644, ACC 0.8060897435897436

Rules for consequent: survived
----------------
IF sex IS female AND sibsp IS Low WITH DS 0.14846236532383247, ACC 0.7337278106508875




## Cross-validation

Because the estimator stores its parameters verbatim, `cross_val_score`
can clone it for every fold. The search settings live in the constructor,
so they are part of what gets cloned.

In [6]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipeline, X, y, cv=folds)
print('fold accuracies:', scores.round(3))
print(f'mean {scores.mean():.3f} +- {scores.std():.3f}')

fold accuracies: [0.805 0.786 0.771 0.756 0.801]
mean 0.784 +- 0.018


## Grid search over the model size and the search budget

`nRules` bounds the model size and `n_gen` the search effort. Both are
ordinary constructor parameters, so `GridSearchCV` can tune them through
the pipeline step name.

In [7]:
grid = GridSearchCV(pipeline, {'fuzzy__nRules': [4, 8], 'fuzzy__n_gen': [15, 30]}, cv=3, n_jobs=1)
grid.fit(X_train, y_train)

results = pd.DataFrame(grid.cv_results_)[['param_fuzzy__nRules', 'param_fuzzy__n_gen', 'mean_test_score', 'std_test_score']]
results.columns = ['nRules', 'n_gen', 'mean cv accuracy', 'std']
print(results.round(3).to_string(index=False))
print('\nbest:', grid.best_params_, f'test accuracy {grid.score(X_test, y_test):.3f}')

 nRules  n_gen  mean cv accuracy   std
      4     15             0.775 0.025
      4     30             0.775 0.025
      8     15             0.778 0.017
      8     30             0.778 0.017

best: {'fuzzy__nRules': 8, 'fuzzy__n_gen': 15} test accuracy 0.817


## Probabilities and explanations on real passengers

In [8]:
best = grid.best_estimator_
passengers = X_test.head(6)
imputed = best['impute'].transform(passengers)
explained = best['fuzzy'].explainable_predict(imputed)
pd.DataFrame({
    'sex': passengers['sex'].to_numpy(), 'age': passengers['age'].to_numpy(), 'pclass': passengers['pclass'].to_numpy(),
    'true': y_test[:6], 'predicted': explained.prediction,
    'p(survived)': best.predict_proba(passengers)[:, list(best.classes_).index('survived')].round(2),
    'winning rule': explained.winning_rule,
})

,sex,age,pclass,true,predicted,p(survived),winning rule
0,male,38.0,3,died,died,0.0,1
1,female,39.0,1,survived,survived,1.0,2
2,female,48.0,3,died,survived,1.0,2
3,female,17.0,3,died,survived,1.0,2
4,male,NaN,3,died,died,0.0,1
5,female,NaN,1,survived,survived,1.0,2


The `winning rule` column indexes the rules printed by `print_rules`, in
order, so every prediction can be traced back to one sentence.